# Suavización exponencial simple y doble de Holt

## Introducción del ejercicio

La suavización exponencial utiliza más peso para las observaciones recientes y menos peso para las observaciones antiguas. Es una técnica útil cuando queremos obtener un pronóstico sencillo, interpretable y rápido.

En este ejercicio analizaremos el **PIB real trimestral**. El problema que buscamos resolver es: **¿cuál podría ser el nivel del PIB real durante los siguientes trimestres y qué diferencia existe entre considerar únicamente el nivel histórico o también su tendencia?**

Compararemos dos modelos:

- **Suavización exponencial simple:** modela únicamente el nivel.
- **Suavización exponencial doble de Holt:** modela el nivel y la tendencia.

El dataset `macrodata` se obtiene desde `statsmodels`, por lo que el notebook puede ejecutarse en Google Colab sin descargar manualmente una fuente externa.

## Objetivos

Al finalizar podremos:

1. Cargar y preparar datos trimestrales desde `statsmodels`.
2. Identificar el nivel y la tendencia de una serie económica.
3. Entrenar suavización exponencial simple.
4. Entrenar suavización exponencial doble de Holt.
5. Comparar ambos modelos con MAE, RMSE y MAPE.
6. Generar un pronóstico de los siguientes cuatro trimestres.
7. Interpretar por qué Holt puede ser más adecuado cuando existe tendencia.

## 1. Instalar y cargar librerías

Google Colab suele incluir estas librerías, pero esta celda asegura que estén disponibles. `statsmodels` proporciona el dataset y los modelos de suavización; `pandas` organiza los datos; `matplotlib` crea visualizaciones y `scikit-learn` calcula métricas.

In [ ]:
%pip install -q statsmodels scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')

## 2. Cargar el dataset `macrodata`

El archivo `macrodata.csv` contiene indicadores macroeconómicos trimestrales. Utilizaremos `realgdp`, el PIB real, como variable objetivo. El usuario subirá el CSV directamente a Google Colab; `statsmodels` se utilizará únicamente para aplicar los modelos de suavización.

In [ ]:
from google.colab import files

archivos_subidos = files.upload()
nombre_archivo = next(iter(archivos_subidos))
macrodata = pd.read_csv(nombre_archivo)

columnas_requeridas = {'year', 'quarter', 'realgdp'}
if not columnas_requeridas.issubset(macrodata.columns):
    faltantes = columnas_requeridas - set(macrodata.columns)
    raise ValueError(f'Faltan columnas requeridas: {faltantes}')

macrodata['fecha'] = pd.PeriodIndex(
    year=macrodata['year'].astype(int),
    quarter=macrodata['quarter'].astype(int),
    freq='Q'
).to_timestamp()

datos = macrodata[['fecha', 'realgdp']].rename(
    columns={'fecha': 'ds', 'realgdp': 'y'}
).set_index('ds').sort_index()
datos.index.name = 'fecha'

print(f'Archivo cargado: {nombre_archivo}')
print(f'Observaciones: {len(datos)}')
print(f'Periodo: {datos.index.min():%Y-%m-%d} a {datos.index.max():%Y-%m-%d}')
datos.head()

## 3. Verificar el archivo cargado

Esta celda confirma que el archivo fue cargado correctamente y que la serie contiene fechas trimestrales y valores válidos de PIB real.

In [ ]:
print(f'Archivo utilizado: {nombre_archivo}')
print(f'Filas cargadas: {len(macrodata):,}')
print(f'Columnas disponibles: {list(macrodata.columns)}')
print(f'Valores faltantes en realgdp: {macrodata["realgdp"].isna().sum()}')

## 4. Explorar la serie y verificar la tendencia

La gráfica permite observar si el PIB real tiene un nivel estable o si presenta una tendencia creciente. Esta observación es importante porque la suavización simple no representa explícitamente una tendencia, mientras que Holt doble sí lo hace.

In [ ]:
plt.figure(figsize=(13, 5))
plt.plot(datos.index, datos['y'], color='#2563eb', linewidth=2)
plt.title('PIB real trimestral')
plt.xlabel('Fecha')
plt.ylabel('PIB real')
plt.tight_layout()

## 5. Separar entrenamiento y prueba

Reservaremos los últimos ocho trimestres para evaluar los pronósticos. La separación mantiene el orden temporal: los modelos solo observarán el pasado y serán evaluados sobre periodos posteriores que no utilizaron para ajustarse.

In [ ]:
horizonte_prueba = 8
entrenamiento = datos.iloc[:-horizonte_prueba].copy()
prueba = datos.iloc[-horizonte_prueba:].copy()

print(f'Entrenamiento: {entrenamiento.index.min():%Y-%m-%d} a {entrenamiento.index.max():%Y-%m-%d}')
print(f'Prueba: {prueba.index.min():%Y-%m-%d} a {prueba.index.max():%Y-%m-%d}')

## 6. Entrenar la suavización exponencial simple

La suavización exponencial simple estima un nivel suavizado. Cada observación reciente recibe más peso que las observaciones antiguas, pero el modelo no calcula una tendencia separada. Por eso, ante una serie que crece, puede quedarse rezagado.

In [ ]:
modelo_simple = SimpleExpSmoothing(
    entrenamiento['y'],
    initialization_method='estimated'
).fit(optimized=True)

pronostico_simple = modelo_simple.forecast(horizonte_prueba)
pronostico_simple.name = 'Suavización simple'
pronostico_simple.head()

## 7. Entrenar la suavización exponencial doble de Holt

Holt agrega un segundo componente: la tendencia. El modelo estima simultáneamente el nivel actual y la dirección del movimiento. En consecuencia, puede extender el crecimiento o disminución observada hacia el futuro. Esta técnica no modela una estacionalidad explícita.

In [ ]:
modelo_holt = Holt(
    entrenamiento['y'],
    initialization_method='estimated'
).fit(optimized=True)

pronostico_holt = modelo_holt.forecast(horizonte_prueba)
pronostico_holt.name = 'Holt doble'
pronostico_holt.head()

## 8. Comparar visualmente ambos pronósticos

La gráfica permite ver la diferencia conceptual: la suavización simple tiende a producir una trayectoria más plana, mientras que Holt doble continúa la tendencia estimada. La línea vertical marca el inicio del periodo de prueba.

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(entrenamiento.index, entrenamiento['y'], label='Entrenamiento', color='#64748b')
plt.plot(prueba.index, prueba['y'], label='Real', color='#111827', linewidth=2)
plt.plot(prueba.index, pronostico_simple, '--', label='Suavización simple', color='#ef4444', linewidth=2)
plt.plot(prueba.index, pronostico_holt, '--', label='Holt doble', color='#16a34a', linewidth=2)
plt.axvline(prueba.index[0], color='black', linestyle=':', label='Inicio de prueba')
plt.title('Comparación de suavización simple y Holt doble')
plt.xlabel('Fecha')
plt.ylabel('PIB real')
plt.legend()
plt.tight_layout()

## 9. Evaluar los errores

Usaremos MAE, RMSE y MAPE. MAE representa el error absoluto promedio; RMSE penaliza más los errores grandes; MAPE expresa el error relativo como porcentaje. En todas las métricas, un valor menor es mejor.

In [ ]:
def calcular_metricas(real, pronostico):
    real = np.asarray(real, dtype=float)
    pronostico = np.asarray(pronostico, dtype=float)
    mascara_valida = np.isfinite(real) & np.isfinite(pronostico) & (real != 0)
    mape = np.mean(np.abs((real[mascara_valida] - pronostico[mascara_valida]) / real[mascara_valida])) * 100
    return pd.Series({
        'MAE': mean_absolute_error(real, pronostico),
        'RMSE': np.sqrt(mean_squared_error(real, pronostico)),
        'MAPE (%)': mape
    })

metricas = pd.DataFrame({
    'Suavización simple': calcular_metricas(prueba['y'], pronostico_simple),
    'Holt doble': calcular_metricas(prueba['y'], pronostico_holt)
}).T
metricas.round(2)

## Interpretación de resultados

Si Holt doble obtiene menores errores, la evidencia indica que capturar la tendencia fue útil durante el periodo de prueba. Si la suavización simple obtiene mejores métricas, puede significar que la tendencia reciente no continuó o que el horizonte fue demasiado largo para extrapolarla.

La elección no debe basarse únicamente en cuál modelo tiene el menor error en una sola partición. En un proyecto real conviene repetir la evaluación en varios periodos temporales y revisar la estabilidad de los resultados.

## 10. Pronosticar los siguientes cuatro trimestres

Después de evaluar los modelos, los ajustaremos nuevamente con toda la historia disponible y pronosticaremos cuatro trimestres. Esto permite aprovechar todos los datos antes de producir el resultado final.

In [ ]:
modelo_simple_final = SimpleExpSmoothing(
    datos['y'], initialization_method='estimated'
).fit(optimized=True)
modelo_holt_final = Holt(
    datos['y'], initialization_method='estimated'
).fit(optimized=True)

horizonte_futuro = 4
futuro_simple = modelo_simple_final.forecast(horizonte_futuro)
futuro_holt = modelo_holt_final.forecast(horizonte_futuro)

fechas_futuras = pd.date_range(
    start=datos.index[-1] + pd.offsets.QuarterBegin(),
    periods=horizonte_futuro,
    freq='QS'
)
pronostico_futuro = pd.DataFrame({
    'Fecha': fechas_futuras,
    'Suavización simple': futuro_simple.to_numpy(),
    'Holt doble': futuro_holt.to_numpy()
})

pronostico_futuro.round(2)

## 11. Interpretar el pronóstico futuro

La diferencia entre ambas trayectorias representa la información adicional de la tendencia. La suavización simple mantiene el último nivel suavizado, mientras que Holt doble proyecta el movimiento estimado. Para decisiones económicas, el resultado debe acompañarse de escenarios y de información externa; ningún modelo garantiza que la tendencia histórica continuará sin cambios.

In [ ]:
plt.figure(figsize=(13, 6))
plt.plot(datos.index, datos['y'], label='Histórico', color='#2563eb', linewidth=2)
plt.plot(pronostico_futuro['Fecha'], pronostico_futuro['Suavización simple'], '--', label='Pronóstico simple', color='#ef4444', linewidth=2)
plt.plot(pronostico_futuro['Fecha'], pronostico_futuro['Holt doble'], '--', label='Pronóstico Holt doble', color='#16a34a', linewidth=2)
plt.axvline(datos.index[-1], color='black', linestyle=':', label='Último dato disponible')
plt.title('Pronóstico de los siguientes cuatro trimestres')
plt.xlabel('Fecha')
plt.ylabel('PIB real')
plt.legend()
plt.tight_layout()

## Diferencia entre suavización simple y Holt doble

- **Suavización exponencial simple:** estima solamente el nivel actual. Es adecuada cuando la serie fluctúa alrededor de un nivel sin tendencia persistente.
- **Holt doble:** estima el nivel y una tendencia. Es adecuada cuando la serie muestra crecimiento o disminución sostenida.
- **Ninguna de las dos modela estacionalidad explícita.** Si existieran patrones repetitivos por trimestre o por año, sería necesario considerar Holt-Winters u otro modelo estacional.

## Conclusiones generales

- La suavización exponencial simple es el modelo más sencillo y sirve como referencia para series sin tendencia marcada.
- Holt doble agrega una estimación de tendencia y puede producir mejores pronósticos cuando el crecimiento histórico continúa.
- La comparación debe realizarse sobre datos futuros no utilizados durante el entrenamiento.
- MAE, RMSE y MAPE ayudan a medir el error, pero la decisión final también depende del contexto económico y del costo de equivocarse.
- Para una serie con tendencia y estacionalidad, Holt doble puede ser insuficiente; en ese caso conviene evaluar Holt-Winters, SARIMA o Prophet.
- El PIB real puede presentar cambios estructurales y ciclos económicos que estos modelos sencillos no explican completamente.